# 01 Agent Path Analysis — detailed Q→answer trace

Purpose: inspect the actual gate behavior on a small set of **local**
questions. For each question this notebook records the **entire LLM call
sequence** (request + response bodies per call) and, for **each gate
decision**, the confidence, relevance, the answer, and the **exact retrieval
text** used to compute that confidence — with the pre-escalation gate flagged.
The full trace is written to `evaluation/notebooks/data/agent_path_trace.txt`.

Input: the first 6 **local** questions of the dev-subset QA set
(`evaluation/data/qa.jsonl`).

Run cells top to bottom. ~6 questions × ~35s + index build ≈ ~4-5 min.

## 1. Setup

Builds the hybrid search index and the agent from `.env` config, via
`evaluation.notebooks.common`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.notebooks.common import build_agent, trace_run, load_qa, sample_qa, setup

setup()
agent, real_call_llm = build_agent()
print("confidence threshold:", agent.confidence_threshold)
print("model:", agent.model)

confidence threshold: 0.5
model: qwen/qwen3.5-9b


## 2. Question set

Only **local** questions (answerable from the local KB): the first 6 of the
dev-subset QA set. Each is tagged `nature=local`, `expected=hybrid -> answer`.

In [2]:
QA = load_qa(PROJECT_ROOT / "evaluation" / "data" / "qa.jsonl")
QUESTIONS = [
    {"question": q["question"], "nature": "local", "expected": "hybrid -> answer"}
    for q in sample_qa(QA, 6)
]
for i, q in enumerate(QUESTIONS, 1):
    print(f"[{i}] ({q['nature']}) {q['question']}")

[1] (local) What are Bulbasaur's two main types and how does that affect its weaknesses to fire and ice moves?
[2] (local) If I want to catch this Seed Pokémon in the wild, what is its capture rate compared to other species?
[3] (local) Which ability acts as Bulbasaur's hidden ability besides Overgrow, and when would it be most useful?
[4] (local) How strong are Grass-type attacks against Bulbasaur given its specific type effectiveness ratios?
[5] (local) Does Bulbasaur evolve into a later stage since the data shows it is not classified as a baby Pokémon?
[6] (local) Ivysaur is a Grass and Poison type, so does it take double damage from Ice moves?


## 3. Run

For each question, run the agent via `trace_run` (which now captures the full
request **and response** body per LLM call) and collect `gate_history` — every
grounding-gate decision with its confidence, relevance, answer, and the
retrieval text it was computed from.

In [3]:
trace_results = {}

for i, q in enumerate(QUESTIONS, 1):
    result, calls, escalated, gate_history = trace_run(agent, q["question"], real_call_llm)
    trace_results[i] = {"question": q["question"], "nature": q["nature"], "result": result, "calls": calls, "gate_history": gate_history}
    status = "rejected" if result.rejected else "accepted"
    print(f"[{i}/{len(QUESTIONS)}] {status} | conf={result.confidence and round(result.confidence, 3)} | calls={len(calls)} | gates={len(gate_history)}")

[1/6] accepted | conf=0.637 | calls=2 | gates=1


[2/6] accepted | conf=0.693 | calls=2 | gates=1


[3/6] accepted | conf=0.598 | calls=2 | gates=1


[4/6] accepted | conf=0.675 | calls=2 | gates=1


[5/6] accepted | conf=0.681 | calls=2 | gates=1


[6/6] accepted | conf=0.544 | calls=2 | gates=1


## 4. Write the detailed trace

Writes a readable per-question block to `evaluation/notebooks/data/agent_path_trace.txt`:
per LLM call the full request + response bodies; per gate decision the
confidence, relevance, answer, and the exact retrieval `search_text` used to
compute that confidence; the pre-escalation gate (the rejected attempt that
triggered escalation) is flagged; and the final outcome.

In [4]:
import json

TRACE_FILE = PROJECT_ROOT / "evaluation" / "notebooks" / "data" / "agent_path_trace.txt"
TRACE_FILE.parent.mkdir(parents=True, exist_ok=True)


def retrieval_text(gate):
    texts = []
    for rec in gate.searches:
        for item in rec.results:
            label = f"id={getattr(item, 'id', '?')} name={getattr(item, 'name', '?')}"
            text = getattr(item, "search_text", "") or getattr(item, "snippet", "")
            texts.append(f"[{label}] {text}")
    return texts


with open(TRACE_FILE, "w", encoding="utf-8") as f:
    for i, t in trace_results.items():
        result = t["result"]
        calls = t["calls"]
        gate_history = t["gate_history"]
        f.write(f"=== Question {i} ===\n")
        f.write(f"Q: {t['question']}\n")
        for j, c in enumerate(calls, 1):
            items = ", ".join(f"({it['type']}, query={it['search_query']!r})" for it in c["items"]) or "no tool call"
            esc = " [ESC]" if c["escalated"] else ""
            f.write(f"--- LLM call {j} ---\n")
            f.write(f"  items: [{items}]{esc}\n")
            f.write(f"  request: {json.dumps(c['request'], ensure_ascii=False, indent=2)}\n")
            f.write(f"  response: {json.dumps(c['response'], ensure_ascii=False, indent=2)}\n")
        for k, g in enumerate(gate_history, 1):
            gs = "rejected" if g.rejected else "accepted"
            pre_esc = g.rejected and (k < len(gate_history))
            f.write(f"--- Gate decision {k} ({gs}, conf={g.confidence and round(g.confidence, 3)}) ---\n")
            f.write(f"  relevance: {g.relevance and round(g.relevance, 3)}\n")
            answer = g.rejected_answer if g.rejected else g.answer
            f.write(f"  answer: {answer}\n")
            f.write("--- Retrieval text for this gate ---\n")
            for txt in retrieval_text(g):
                f.write(f"  {txt}\n")
            if pre_esc:
                f.write(f">>> PRE-ESCALATION GATE (rejected, conf={g.confidence and round(g.confidence, 3)}) <<<\n")
        status = "rejected" if result.rejected else "accepted"
        f.write(f"--- Final outcome: {status}, source={result.source}, confidence={result.confidence and round(result.confidence, 3)}, relevance={result.relevance and round(result.relevance, 3)} ---\n")
        f.write(f"  answer: {result.answer}\n")
        f.write("\n")

print(f"wrote {len(trace_results)} questions to {TRACE_FILE}")

wrote 6 questions to /Volumes/wd_external_drive/yingzhang/Courses/llm-zoomcamp-2026/llm_zoomcamp_project/evaluation/notebooks/data/agent_path_trace.txt


## 5. Summary

Minimal per-question outcome at a glance (the full detail is in the trace
file).

In [5]:
import pandas as pd

rows = []
for i, t in trace_results.items():
    r = t["result"]
    rows.append({
        "#": i,
        "nature": t["nature"],
        "question": t["question"],
        "accepted": not r.rejected,
        "source": r.source,
        "confidence": round(r.confidence, 3) if r.confidence else None,
        "relevance": round(r.relevance, 3) if r.relevance is not None else None,
        "llm calls": len(t["calls"]),
        "gates": len(t["gate_history"]),
    })
df = pd.DataFrame(rows)
df

,#,nature,question,accepted,source,confidence,relevance,llm calls,gates
0,1,local,What are Bulbasaur's two main types and how do...,True,local,0.637,0.863,2,1
1,2,local,If I want to catch this Seed Pokémon in the wi...,True,local,0.693,0.685,2,1
2,3,local,Which ability acts as Bulbasaur's hidden abili...,True,local,0.598,0.724,2,1
3,4,local,How strong are Grass-type attacks against Bulb...,True,local,0.675,0.866,2,1
4,5,local,Does Bulbasaur evolve into a later stage since...,True,local,0.681,0.872,2,1
5,6,local,"Ivysaur is a Grass and Poison type, so does it...",True,local,0.544,0.926,2,1
